# 04 · Cleaning & preparation

**Phase skill: turning messy text into structured data, and measuring how well you did.**

Every Shadowdark monster carries one free-text attack line. The 15 strings below are
real rows from `sd_monsters`, chosen to cover the whole mess: simple attacks,
multi-clause lines joined by `and`/`or`, ranged parentheticals, a negative bonus, rider
text inside the damage, and one string that is genuinely unparseable.

The format, in words: each **clause** looks like
`<count> <name> [(range)] [+/-bonus] [(damage)]`. The damage parenthetical can hold a
dice expression like `2d6`, flat numbers, and/or rider words (e.g. `poison`), all
joined by `+`.

**Do not open `src/parse_stats.py` yet.** It is the pipeline's answer to this exact
problem, and you get far more out of it after you've fought the regex yourself.

In [ ]:
RAW_ATTACKS = [
    ("Badger", "2 claw +2 (1d4)"),
    ("Bat, Giant", "1 bite +2 (1d6)"),
    ("Archangel", "3 flaming greatsword +10 (2d12)"),
    ("Archmage", "2 spell +7"),
    ("Assassin", "2 poisoned dagger (close/near) +6 (2d4)"),
    ("Deep One", "2 spear (close/ near) +2 (1d6)"),
    ("Beastman", "1 spear (close/near) +2 (1d6 + 1)"),
    ("Basilisk", "2 bite +4 (2d6 + petrify)"),
    ("Acolyte", "1 mace +1 (1d6) or 1 spell +2"),
    ("Angel, Domini", "3 bastard sword +7 (1d8) or 1 horn"),
    ("Dryad", "1 staff -1 (1d4) or 1 charm"),
    ("Cave Brute", "2 claw +5 (1d8) and 1 mandible +5 (1d10)"),
    ("Aboleth", "2 tentacle (near) +5 (1d8 + curse) or 1 tail +5 (3d6)"),
    ("Demon, Balor", "3 greatsword +10 (2d12 + hellfire) and 1 fire whip (near) +10 (2d6 + grab)"),
    ("Kraken", "4 tentacle (near) +9 (2d12) or 1 storm or 1d4 lightning bolt"),
]
print(f"{len(RAW_ATTACKS)} raw attack strings")

## Exercise 4.1 — split into clauses

A line like `1 mace +1 (1d6) or 1 spell +2` is two separate attack options. Split every
raw string on the connector words `and` / `or` (with spaces around them) and collect the
pieces.

**Produce:** `clauses` — a list of `(monster_name, clause)` tuples, each clause stripped
of surrounding whitespace. Careful: `(close/near)` contains no connector and must stay
intact.

<details><summary>Hint 1 (concept)</summary>

Splitting on a word is different from splitting on a character -- the connector is only a connector when it stands alone between spaces.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `re.split` with an alternation pattern.

</details>

In [ ]:
# Produce: clauses (list of (monster_name, clause) tuples)

clauses = ...

len(clauses)

In [ ]:
assert len(clauses) == 23, f"expected 23 clauses from the 15 strings, got {len(clauses)}"
per = {}
for monster, clause in clauses:
    per.setdefault(monster, []).append(clause)
assert len(per["Kraken"]) == 3, f"Kraken should split into 3 clauses, got {len(per['Kraken'])}"
assert len(per["Assassin"]) == 1, "Assassin is a single clause -- '(close/near)' must not trigger a split"
assert per["Dryad"][0] == "1 staff -1 (1d4)", (
    f"Dryad's first clause is {per['Dryad'][0]!r} -- clauses should carry no surrounding spaces")
print("PASS: 23 clauses, splits and stripping all correct.")
print("Context: most text cleaning starts exactly here -- find the record separator hiding inside a string.")

## Exercise 4.2 — parse one clause

Write the parser. `parse_attack_clause(clause)` returns a 4-tuple
`(num_attacks, attack_name, attack_bonus, avg_damage)`, or `None` if the clause doesn't
fit the format. The contract:

- `num_attacks` — leading count, `int`.
- `attack_name` — the name words only: the range parenthetical (e.g. `(close/near)`)
  is **not** part of the name.
- `attack_bonus` — `int`, may be negative; `None` when the clause lists no bonus.
- `avg_damage` — `float`: the dice average (`NdM` averages `N * (M + 1) / 2`) **plus**
  any flat number in the damage parenthetical. Rider words (`petrify`, `grab`, …) add
  nothing. `None` when there is no damage parenthetical at all.
- A clause that doesn't start with a count-then-name (like Kraken's third one) → `None`.

**Produce:** the function `parse_attack_clause`.

<details><summary>Hint 1 (concept)</summary>

Treat the clause as fixed slots -- count, name words, optional range paren, optional signed bonus, optional damage paren -- and match the slots in order instead of hunting substrings.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `re.match` with named groups, optional groups (`?`), and a signed-integer pattern like `[-+]\d+`.

</details>

In [ ]:
def parse_attack_clause(clause):
    """Return (num_attacks, attack_name, attack_bonus, avg_damage) or None."""
    ...

In [ ]:
cases = {
    "2 claw +2 (1d4)": (2, "claw", 2, 2.5),
    "3 flaming greatsword +10 (2d12)": (3, "flaming greatsword", 10, 13.0),
    "2 spell +7": (2, "spell", 7, None),
    "2 poisoned dagger (close/near) +6 (2d4)": (2, "poisoned dagger", 6, 5.0),
    "2 spear (close/ near) +2 (1d6)": (2, "spear", 2, 3.5),
    "1 spear (close/near) +2 (1d6 + 1)": (1, "spear", 2, 4.5),
    "1 staff -1 (1d4)": (1, "staff", -1, 2.5),
    "1 horn": (1, "horn", None, None),
    "2 tentacle (near) +5 (1d8 + curse)": (2, "tentacle", 5, 4.5),
    "1 fire whip (near) +10 (2d6 + grab)": (1, "fire whip", 10, 7.0),
}
for s, want in cases.items():
    got = parse_attack_clause(s)
    assert got == want, f"parse_attack_clause({s!r}) -> {got!r}, expected {want!r}"
assert parse_attack_clause("1d4 lightning bolt") is None, (
    "'1d4 lightning bolt' does not fit the count-then-name format -- it should return None, "
    f"not {parse_attack_clause('1d4 lightning bolt')!r}")
print(f"PASS: all {len(cases)} parseable cases plus the unparseable one.")
print("Context: the contract cases cover every wrinkle in the full dataset -- range parens, riders, flat bonuses, negatives.")

## Exercise 4.3 — your parse success rate

Run your parser over **all 23 clauses** from 4.1.

**Produce:**

- `parsed` — list of the successful results (the non-`None` returns),
- `success_rate` — fraction of the 23 clauses parsed, as a float in 0–1.

<details><summary>Hint 1 (concept)</summary>

A cleaning step without a measured success rate is an anecdote. The failures matter as much as the rate -- know which clause fails and why.

</details>
<details><summary>Hint 2 (what to look up)</summary>

A list comprehension with an `is not None` filter does both counts.

</details>

In [ ]:
# Produce: parsed (list of tuples), success_rate (float 0-1)

parsed = ...
success_rate = ...

print(f"{len(parsed)} of {len(clauses)} clauses parsed ({success_rate:.1%})")

In [ ]:
assert len(parsed) == 22, f"parsed holds {len(parsed)} results -- exactly one of the 23 clauses is genuinely unparseable"
assert abs(success_rate - 22 / 23) < 1e-6, (
    f"success_rate = {success_rate:.4f} -- it should be the parsed fraction of all 23 clauses")
print(f"PASS: success rate {success_rate:.1%} -- and you know exactly which clause failed and why.")
print("Context: the full pipeline parses 358 of 359 clauses (99.7%) across all 243 monsters -- "
      "defeated by the same lone Kraken clause that defeated you.")

## Now — and only now — read the official answer

Open [`src/parse_stats.py`](../src/parse_stats.py) and compare it with what you wrote:

- How does its clause regex handle the slots you fought with?
- What does it do with rider text instead of discarding it? (Look at the `sd_attacks`
  schema.)
- What happens to unparseable clauses — a crash, a skip, or something better?

Note the pipeline does **not** chase 100%. It logs the failure to a file and moves on.
A cleaning step that knows and records its own failure rate beats one that silently
"handles" everything.